# Pipeline de Consolidação e Alinhamento Temporal de Séries Históricas (B3)
**Autor:** Pedro Reis  
**Escopo:** Processamento de Dados Intermediários para Análise de Portfólios (MPT, PMPT, BL)

Este notebook implementa o pipeline automatizado de ingestão, limpeza, alinhamento temporal e exportação multi-formato das séries históricas de preços de fechamento obtidas via Economatica. 

### Objetivos Metodológicos:
1. **Isolamento de Escopo:** Leitura seletiva de arquivos `.xlsx` na pasta `\data\dados_economatica`.
2. **Sincronização Cronológica:** Alinhamento de todos os ativos por uma matriz de datas única.
3. **Filtro de Horizonte Temporal:** Delimitação estrita entre **01/01/2010** e **31/12/2025**.
4. **Tratamento de Missing Values:** Mitigação de lacunas por meio do método *Forward Fill* ($ffill$).
5. **Persistência Eficiente:** Salvamento unificado em `.csv` e `.parquet` em `\data\dados_economatica_combinados`.

In [6]:
%pip uninstall pyarrow
%pip install pyarrow

^C
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import glob
import logging
import pathlib
from pathlib import Path
import pandas as pd


# Configuração do sistema de logs para monitoramento do pipeline em lote
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)


# Definição dos caminhos estritos do projeto utilizando caminhos relativos robustos

DIRETORIO_ORIGEM = Path(r"C:\VSCodeWorkspace\TCC_Final\data\dados_economatica")
DIRETORIO_DESTINO = Path(r"C:\VSCodeWorkspace\TCC_Final\data\dados_economatica_tratados")

# Garantir a criação da pasta de destino caso não exista em disco
DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

print(f"Diretório de origem mapeado: {DIRETORIO_ORIGEM}")
print(f"Diretório de destino estruturado: {DIRETORIO_DESTINO.resolve()}")

Diretório de origem mapeado: C:\VSCodeWorkspace\TCC_Final\data\dados_economatica
Diretório de destino estruturado: C:\VSCodeWorkspace\TCC_Final\data\dados_economatica_tratados


## Ingestão de Dados e Extração Individual com Schema Estrito

Nesta etapa, o algoritmo realiza a varredura do diretório de origem buscando arquivos com a extensão `.xlsx`. Para cada arquivo localizado, é feita uma leitura parametrizada via Pandas:
* `skiprows=3`: Descarte automático das três primeiras linhas que contêm metadados textuais da Economatica.
* `usecols=[0, 4]`: Ingestão restrita apenas às colunas de interesse posicional: Índice 0 ('Data') e Índice 4 ('Fechamento').

In [20]:
# Mapeamento vetorial de todos os arquivos Excel na pasta de origem
arquivos_excel = list(DIRETORIO_ORIGEM.glob("*.xlsx"))
print(f"Total de relatórios identificados para processamento: {len(arquivos_excel)}")

# Lista para armazenamento dos DataFrames individuais preparados
lista_dataframes = []

for caminho_arquivo in arquivos_excel:
    nome_arquivo = caminho_arquivo.name
    
    # Extração e isolamento do Ticker do ativo contido no nome do arquivo padrão Economatica
    # Exemplo: 'Economatica-correcao_dividendos-AGRO3.xlsx' vira 'AGRO3'
    ticker = nome_arquivo.replace('Economatica-correcao_dividendos-', '').replace('.xlsx', '').strip()

    if not nome_arquivo.startswith('~$'):
        try:
            # Leitura performática contendo apenas o schema necessário
            df_individual = pd.read_excel(
                caminho_arquivo,
                skiprows=3,
                usecols=lambda c: c in ['Data', 'Fechamento'],
                na_values=['-'],
                keep_default_na=True
            )
            
            # Validação defensiva do schema mínimo exigido
            if 'Data' not in df_individual.columns or 'Fechamento' not in df_individual.columns:
                logging.warning(f"Arquivo rejeitado por desconformidade de schema: {nome_arquivo}")
                continue
                
            # Coerção explícita de tipos de dados para prevenção de anomalias
            df_individual['Data'] = pd.to_datetime(df_individual['Data'], errors='coerce')
            df_individual['Fechamento'] = pd.to_numeric(df_individual['Fechamento'], errors='coerce')
            
            # Expulsão de registros com chaves temporais nulas
            df_individual = df_individual.dropna(subset=['Data'])
            
            # Renomeação da coluna de preços para o Ticker correspondente para evitar colisão na junção
            df_individual = df_individual.rename(columns={'Fechamento': ticker})
            
            # Definição da Data como índice estrutural para o alinhamento
            df_individual = df_individual.set_index('Data')
            
            lista_dataframes.append(df_individual)
            
        except Exception as e:
            logging.error(f"Falha na leitura do arquivo {nome_arquivo}. Erro: {str(e)}")

print(f"Fase concluída. Total de ativos convertidos com sucesso: {len(lista_dataframes)}")

Total de relatórios identificados para processamento: 497
Fase concluída. Total de ativos convertidos com sucesso: 496


## Concatenação Dimensional, Filtro Temporal e Normalização de Janelas

Com todos os DataFrames individuais indexados por data, efetuamos a junção geométrica lateral (`axis=1`). O Pandas alinhará automaticamente as linhas correspondentes às mesmas datas. 

Na sequência, o script aplica:
1. **Filtro Temporal:** Recorte estrito entre `2010-01-01` e `2025-12-31`.


In [ ]:
print("Iniciando a concatenação global das séries...")
# Fusão de todas as matrizes baseada no índice cronológico comum
df_consolidado = pd.concat(lista_dataframes, axis=1, sort=False)

# Ordenar o índice de forma estritamente cronológica crescente
df_consolidado = df_consolidado.sort_index()

# Definição das datas limítrofes do horizonte temporal do TCC
data_inicio = pd.to_datetime('2010-01-01')
data_fim = pd.to_datetime('2025-12-31')

# Aplicação do filtro de corte temporal
df_recortado = df_consolidado.loc[data_inicio:data_fim].copy()


Iniciando a concatenação global das séries...


C:\Users\01834179173\AppData\Local\Temp\ipykernel_9444\2953323430.py:3: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_consolidado = pd.concat(lista_dataframes, axis=1)


## Persistência de Dados e Exportação Redundante

A matriz bidimensional tratada e sincronizada é salva no diretório de destino sob dois formatos distintos:
1. **CSV (`dados_combinados_tcc.csv`):** Formato universal em texto plano para máxima interoperabilidade.
2. **Parquet (`dados_combinados_tcc.parquet`):** Formato binário colunar de alto desempenho, altamente compactado e que preserva de forma nativa os tipos de dados binários em disco.

In [9]:
# Definição das rotas e nomes finais dos arquivos consolidados
caminho_saida_csv = DIRETORIO_DESTINO / "dados_combinados_tcc.csv"
caminho_saida_parquet = DIRETORIO_DESTINO / "dados_combinados_tcc.parquet"

try:
    # Salvamento da matriz em formato CSV
    df_recortado.to_csv(caminho_saida_csv, index=True)
    print(f"Sucesso: Matriz CSV persistida em {caminho_saida_csv.name}")
    
    # Salvamento da matriz em formato Parquet (colunar otimizado)
    df_recortado.to_parquet(caminho_saida_parquet, index=True, engine='pyarrow')
    print(f"Sucesso: Matriz Parquet persistida em {caminho_saida_parquet.name}")
    
    print("\n--- PIPELINE EXECUTADO E FINALIZADO COM SUCESSO ---")
    print(f"Dimensões finais da matriz (Linhas [Datas] x Colunas [Ativos]): {df_tratado.shape}")

except Exception as e:
    logging.critical(f"Erro catastrófico na persistência dos dados consolidados: {str(e)}")

2026-05-28 12:41:33,462 [CRITICAL] - Erro catastrófico na persistência dos dados consolidados: `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.


Sucesso: Matriz CSV persistida em dados_combinados_tcc.csv


================================================================================
APÊNDICE A – SCRIPT DE INGESTÃO, ALINHAMENTO TEMPORAL E CONSOLIDAÇÃO DE DADOS
================================================================================

Este apêndice documenta o algoritmo computacional desenvolvido em linguagem Python 
para automatizar a etapa de extração, transformação, alinhamento cronológico e 
carga (ETL) das séries históricas de preços dos ativos listados na B3. Os dados 
brutos individuais foram extraídos originalmente da plataforma Economatica.

O script foi projetado para garantir o rigor metodológico da pesquisa, operando 
sob as seguintes premissas técnicas e restrições analíticas:
1. Eliminação manual de cabeçalhos e metadados descritivos contidos nas três 
   primeiras linhas de cada relatório padrão emitido pela plataforma de origem.
2. Restrição dimensional volumétrica via carregamento seletivo de vetores, 
   capturando de forma exclusiva as colunas "Data" e "Fechamento".
3. Sincronização temporal síncrona através do método de concatenação dimensional 
   ancorado no índice cronológico de datas comuns.
4. Aplicação de filtro temporal estrito delimitando o horizonte amostral da 
   pesquisa entre 01/01/2010 e 31/12/2025.
5. Exportação redundante da matriz bidimensional resultante em formato textual 
   (CSV).

O código fonte integral, estruturado para execução em células de cadernos 
interativos (Jupyter Notebooks), encontra-se transcrito na íntegra a seguir:

[INSERIR O CÓDIGO DAS CÉLULAS 2, 4, 6 e 8 NESTE ESPAÇO]

import os
import glob
import logging
from pathlib import Path
import pandas as pd

# Configuração do sistema de logs para monitoramento do pipeline em lote
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] - %(message)s"
)

# Definição dos caminhos estritos do projeto utilizando caminhos relativos robustos
DIRETORIO_ORIGEM = Path("./data/dados_economatica")
DIRETORIO_DESTINO = Path("./data/dados_economatica_combinados")

# Garantir a criação da pasta de destino caso não exista em disco
DIRETORIO_DESTINO.mkdir(parents=True, exist_ok=True)

print(f"Diretório de origem mapeado: {DIRETORIO_ORIGEM.resolve()}")
print(f"Diretório de destino estruturado: {DIRETORIO_DESTINO.resolve()}")

# Mapeamento vetorial de todos os arquivos Excel na pasta de origem
arquivos_excel = list(DIRETORIO_ORIGEM.glob("*.xlsx"))
print(f"Total de relatórios identificados para processamento: {len(arquivos_excel)}")

# Lista para armazenamento dos DataFrames individuais preparados
lista_dataframes = []

for caminho_arquivo in arquivos_excel:
    nome_arquivo = caminho_arquivo.name
    
    # Extração e isolamento do Ticker do ativo contido no nome do arquivo padrão Economatica
    # Exemplo: 'Economatica-correcao_dividendos-AGRO3.xlsx' vira 'AGRO3'
    ticker = nome_arquivo.replace('Economatica-correcao_dividendos-', '').replace('.xlsx', '').strip()
    
    try:
        # Leitura performática contendo apenas o schema necessário
        df_individual = pd.read_excel(
            caminho_arquivo,
            skiprows=3,
            usecols=[0, 4],
            na_values=['-'],
            keep_default_na=True
        )
        
        # Validação defensiva do schema mínimo exigido
        if 'Data' not in df_individual.columns or 'Fechamento' not in df_individual.columns:
            logging.warning(f"Arquivo rejeitado por desconformidade de schema: {nome_arquivo}")
            continue
            
        # Coerção explícita de tipos de dados para prevenção de anomalias
        df_individual['Data'] = pd.to_datetime(df_individual['Data'], errors='coerce')
        df_individual['Fechamento'] = pd.to_numeric(df_individual['Fechamento'], errors='coerce')
        
        # Expulsão de registros com chaves temporais nulas
        df_individual = df_individual.dropna(subset=['Data'])
        
        # Renomeação da coluna de preços para o Ticker correspondente para evitar colisão na junção
        df_individual = df_individual.rename(columns={'Fechamento': ticker})
        
        # Definição da Data como índice estrutural para o alinhamento
        df_individual = df_individual.set_index('Data')
        
        lista_dataframes.append(df_individual)
        
    except Exception as e:
        logging.error(f"Falha na leitura do arquivo {nome_arquivo}. Erro: {str(e)}")

print(f"Fase concluída. Total de ativos convertidos com sucesso: {len(lista_dataframes)}")

print("Iniciando a concatenação global das séries...")
# Fusão de todas as matrizes baseada no índice cronológico comum
df_consolidado = pd.concat(lista_dataframes, axis=1)

# Ordenar o índice de forma estritamente cronológica crescente
df_consolidado = df_consolidado.sort_index()

# Definição das datas limítrofes do horizonte temporal do TCC
data_inicio = pd.to_datetime('2010-01-01')
data_fim = pd.to_datetime('2025-12-31')

# Aplicação do filtro de corte temporal
df_recortado = df_consolidado.loc[data_inicio:data_fim].copy()

# Contagem diagnóstica inicial de dados faltantes (NaN) pré-normalização
nulos_antes = df_recortado.isna().sum().sum()
print(f"Quantidade total de NaNs identificados no intervalo: {nulos_antes}")

# Tratamento de missing values: Propagação do último preço válido para frente (Forward Fill)
df_tratado = df_recortado.ffill()

# Tratamento complementar residual (Backward Fill) caso a primeira linha possua NaNs
df_tratado = df_tratado.bfill()

nulos_depois = df_tratado.isna().sum().sum()
print(f"Quantidade residual de NaNs pós-tratamento de preenchimento: {nulos_depois}")

# Definição das rotas e nomes finais dos arquivos consolidados
caminho_saida_csv = DIRETORIO_DESTINO / "dados_combinados_tcc.csv"
caminho_saida_parquet = DIRETORIO_DESTINO / "dados_combinados_tcc.parquet"

try:
    # Salvamento da matriz em formato CSV
    df_tratado.to_csv(caminho_saida_csv, index=True)
    print(f"Sucesso: Matriz CSV persistida em {caminho_saida_csv.name}")
    
    # Salvamento da matriz em formato Parquet (colunar otimizado)
    df_tratado.to_parquet(caminho_saida_parquet, index=True, engine='pyarrow')
    print(f"Sucesso: Matriz Parquet persistida em {caminho_saida_parquet.name}")
    
    print("\n--- PIPELINE EXECUTADO E FINALIZADO COM SUCESSO ---")
    print(f"Dimensões finais da matriz (Linhas [Datas] x Colunas [Ativos]): {df_tratado.shape}")

except Exception as e:
    logging.critical(f"Erro catastrófico na persistência dos dados consolidados: {str(e)}")